# Flying a Quadcopter in Simulation

Companion notebook for the [SimWeave blog post](https://notabot123.github.io/simweave/blog/quadcopter-flight/).

This notebook covers:
- The `Quadrotor` model and the `Mixer`
- A take-off, figure-8, landing `Mission` flown with `fly(...)`
- The sampled-data controller hook underneath `fly`
- 3D path and flight summary plots
- Wind, an unknown payload and a weak motor
- A small Monte Carlo of landing dispersion
- Progressive motor failure with `FaultInjector`

Needs `simweave>=0.9.0` with the `viz` extra.

In [1]:
# Uncomment to install if running outside Binder
# !pip install 'simweave[viz]>=0.9.0' kaleido

import numpy as np
import simweave
from simweave.flight import (
    CascadedController, ConstantWind, GustWind, Mission, Mixer, Quadrotor,
    TurbulentWind, fly,
)

print("simweave", simweave.__version__)

simweave 0.9.0


## 1  The vehicle

16 states: position, velocity, Euler angles, body rates and the four rotor thrusts (which lag the commands).
The world is **Z up**; positive pitch is nose down.

In [2]:
quad = Quadrotor()
print("hover thrust per rotor", round(quad.hover_thrust, 3), "N")
print("max thrust per rotor  ", quad.max_thrust, "N")
print("thrust-to-weight      ", round(quad.params.thrust_to_weight, 2))

hover thrust per rotor 2.943 N
max thrust per rotor   7.0 N
thrust-to-weight       2.38


In [3]:
mixer = Mixer(quad.params)
print(np.round(mixer.allocate(quad.params.weight, (0.0, 0.0, 0.0)), 3))   # plain hover
print(np.round(mixer.allocate(quad.params.weight, (0.02, 0.0, 0.0)), 3))  # a little roll torque

[2.943 2.943 2.943 2.943]
[2.901 2.985 2.985 2.901]


## 2  Your first mission

In [4]:
mission = Mission.figure8(height=3.0, a=4.0, b=2.0, period=20.0)
for name, start, end in mission.phases:
    print(f"{name:<10s} {start:5.1f} -> {end:5.1f} s")

flight = fly(Quadrotor(), mission)
print({k: round(v, 3) for k, v in flight.metrics().items()})
print({k: round(v, 3) for k, v in flight.phase_rms().items()})

takeoff      0.0 ->   4.0 s
hover        4.0 ->   6.0 s
transition   6.0 ->   9.0 s
figure8      9.0 ->  29.0 s
transition  29.0 ->  32.0 s
hover       32.0 ->  34.0 s
land        34.0 ->  40.0 s
landed      40.0 ->  42.0 s


{'rms_error': 0.018, 'max_error': 0.072, 'max_tilt_deg': 15.594, 'saturation_fraction': 0.0, 'energy_wh': 1.027, 'duration': 42.0, 'touchdown_speed': -0.397, 'landing_error': 0.001}
{'takeoff': 0.025, 'hover': 0.016, 'transition': 0.032, 'figure8': 0.013, 'land': 0.003, 'landed': 0.001}


In [5]:
data = flight.to_dict()          # time, x, y, z, roll, pitch, yaw, ... (Z up, radians)
print(list(data)[:7], len(data["time"]), "samples")
viewer = flight.to_viewer()      # same, mapped for a Y-up 3D viewer
print(list(viewer))

['time', 'x', 'y', 'z', 'roll', 'pitch', 'yaw'] 10501 samples
['time', 'x', 'y', 'z', 'roll', 'pitch', 'yaw']


## 3  Under the hood: the sampled-data controller hook

`fly` builds a `CascadedController` and hands it to `simulate`. The controller runs every `control_dt` seconds
and its output is held while the physics integrates in finer steps.

In [6]:
from simweave.continuous import simulate

quad = Quadrotor()
ctrl = CascadedController(mission, quad.params)
result = simulate(quad, (0.0, mission.duration), dt=0.004, controller=ctrl, control_dt=0.008)
print(result.state.shape, result.state_labels[:6], "final z", round(float(result.state[-1, 2]), 3))

(10501, 16) ('x', 'y', 'z', 'vx', 'vy', 'vz') final z 0.0


## 4  Seeing the flight

Flown in a steady 4 m/s wind.

In [7]:
from simweave.viz import plot_trajectory_3d, plot_flight_summary

windy = fly(Quadrotor(wind=ConstantWind((4.0, 0.0, 0.0))), mission)
fig_path = plot_trajectory_3d(windy)
fig_path.show()

In [8]:
fig_summary = plot_flight_summary(windy)
fig_summary.show()

## 5  Wind, payloads and weak motors

The controller models the vehicle with `vehicle.params`. Changing the vehicle afterwards tests it against
model mismatch: a parcel nobody weighed, a motor that is wearing out.

In [9]:
def run(quad):
    m = fly(quad, mission).metrics()
    return m["rms_error"], m["max_error"], m["energy_wh"]

heavy = Quadrotor()
heavy.mass *= 1.25          # a payload the controller does not know about

weak = Quadrotor()
weak.motor_eff_1 = 0.8      # rotor 1 delivers only 80 % of commanded thrust

cases = {
    "nominal": Quadrotor(),
    "5 m/s steady wind": Quadrotor(wind=ConstantWind((5.0, 0.0, 0.0))),
    "6 m/s gust": Quadrotor(wind=GustWind((6.0, 0.0, 0.0), start=15.0, duration=3.0)),
    "payload +25 %": heavy,
    "rotor 1 at 80 %": weak,
}
print(f"{'case':<24s} {'rms (m)':>8s} {'max (m)':>8s} {'energy (Wh)':>12s}")
for name, quad in cases.items():
    rms, mx, wh = run(quad)
    print(f"{name:<24s} {rms:8.3f} {mx:8.3f} {wh:12.2f}")

case                      rms (m)  max (m)  energy (Wh)


nominal                     0.018    0.072         1.03


5 m/s steady wind           0.025    0.072         1.03


6 m/s gust                  0.023    0.087         1.03


payload +25 %               0.061    0.239         1.44


rotor 1 at 80 %             0.035    0.148         1.13


## 6  Monte Carlo: where will it land?

Random steady wind, seeded turbulence and an unknown payload. Here the runner is serial so the scenario can be
defined in the notebook; in a script, define it at module level and use `executor="processes"`
(see `demos/28_quadcopter_wind_montecarlo.py`).

In [10]:
from simweave.mc import run_monte_carlo
from simweave.viz import plot_landing_dispersion

MISSION = Mission.figure8(height=3.0, period=20.0)

def scenario(seed):
    rng = np.random.default_rng(seed)
    speed, heading = rng.uniform(0, 8), rng.uniform(0, 2 * np.pi)
    wind = TurbulentWind(
        sigma=1.0, correlation_time=1.5, duration=MISSION.duration + 1.0,
        mean=(speed * np.cos(heading), speed * np.sin(heading), 0.0), rng=rng,
    )
    quad = Quadrotor(wind=wind)
    quad.mass *= rng.uniform(0.9, 1.3)
    flight = fly(quad, MISSION)
    td = flight.touchdown()
    return [td["x"], td["y"], td["vertical_speed"], flight.metrics()["rms_error"]]

mc = run_monte_carlo(scenario, n_runs=12, executor="serial")
samples = np.asarray(mc.samples)
miss = np.hypot(samples[:, 0], samples[:, 1])
print(f"RMS tracking error  median {np.median(samples[:, 3]):.3f} m, worst {samples[:, 3].max():.3f} m")
print(f"Touchdown speed     median {np.median(samples[:, 2]):.2f} m/s, hardest {samples[:, 2].min():.2f} m/s")
print(f"Landing miss        50% within {np.quantile(miss, 0.5):.3f} m, 95% within {np.quantile(miss, 0.95):.3f} m")

RMS tracking error  median 0.037 m, worst 0.059 m
Touchdown speed     median -0.43 m/s, hardest -0.59 m/s
Landing miss        50% within 0.012 m, 95% within 0.035 m


In [11]:
fig_landing = plot_landing_dispersion(samples[:, :2], title="Landing dispersion: 12 flights")
fig_landing.show()

## 7  When a motor fails

`FaultInjector` degrades `motor_eff_1` between 12 s and 26 s. The controller does not know; its integrators push
the weak rotor harder until it runs out of thrust.

In [12]:
import plotly.graph_objects as go
from simweave.faults import FaultInjector, FaultProfile, ParameterFault

def fly_with_loss(loss):
    quad = Quadrotor()
    if loss == 0.0:
        return fly(quad, mission)
    profile = FaultProfile(onset_time=12.0, failure_time=26.0, mode="rotor_degradation", shape="linear")
    fault = ParameterFault(param="motor_eff_1", profile=profile, max_delta=-loss)
    return fly(FaultInjector(quad, [fault]), mission)

fig_fault = go.Figure()
print(f"{'case':<10s} {'rms':>7s} {'max':>6s} {'saturated':>10s} {'energy Wh':>10s}  outcome")
for name, loss in {"healthy": 0.0, "50 % loss": 0.5, "60 % loss": 0.6}.items():
    f = fly_with_loss(loss)
    m = f.metrics()
    lowest = f.position[(f.time > 8) & (f.time < 34), 2].min()
    outcome = f"lands at {-m['touchdown_speed']:.2f} m/s" if lowest > 0.5 else "crash"
    print(f"{name:<10s} {m['rms_error']:7.3f} {m['max_error']:6.2f} {m['saturation_fraction']:10.0%} "
          f"{m['energy_wh']:10.2f}  {outcome}")
    fig_fault.add_trace(go.Scatter(x=f.time, y=f.position[:, 2], name=name))
fig_fault.add_vrect(x0=12, x1=26, fillcolor="grey", opacity=0.12, line_width=0,
                    annotation_text="rotor 1 degrading")
fig_fault.update_layout(title="Altitude under rotor 1 degradation",
                        xaxis_title="time (s)", yaxis_title="altitude (m)")
fig_fault.show()

case           rms    max  saturated  energy Wh  outcome


healthy      0.018   0.07         0%       1.03  lands at 0.40 m/s


50 % loss    0.072   0.18         0%       1.25  lands at 0.31 m/s


60 % loss    1.331   3.02        36%       1.33  crash


The cliff has a closed form: the weakened rotor must still deliver hover thrust, so it saturates once it has lost
`1 - hover_thrust / max_thrust` of its thrust.

In [13]:
q = Quadrotor()
print(round(1 - q.hover_thrust / q.max_thrust, 2))

0.58


## 8  Export the figures

Writes PNGs to `img/` (needs Kaleido).

In [14]:
import os
os.makedirs("img", exist_ok=True)
for name, fig, size in (
    ("flight_path3d", fig_path, (1000, 700)),
    ("flight_summary", fig_summary, (1000, 700)),
    ("flight_landing", fig_landing, (900, 650)),
    ("flight_fault_altitude", fig_fault, (900, 500)),
):
    fig.write_image(f"img/{name}.png", width=size[0], height=size[1])
print("saved", sorted(f for f in os.listdir("img") if f.startswith("flight_")))

saved ['flight_fault_altitude.png', 'flight_landing.png', 'flight_path3d.png', 'flight_summary.png']
